# 02 - Feature engineering and feature sets

Rubric: *20+ new features with descriptions* and *5+ generated transformations*. All features are computed in
`setalpha/transform.py`; each uses only information known at the Thai close of that day (US / FRED / Cboe data
lagged one day).

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from setalpha import config, ingest, transform, model, regime, portfolio, strategy, evaluate, report, pipeline
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:.4f}".format)
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True})

In [ ]:
df = transform.add_dummies(transform.load())
sets = transform.feature_sets(df)
print({k: len(v) for k, v in sets.items()})
print("lecture-style features:", len(transform.LECTURE_FEATURES), "| new features:", len(transform.NEW_FEATURES),
      "| dummy columns:", len(sets["DUMMIES"]))

## Feature dictionary (new features)

| Group | Feature | Definition | Why it could predict alpha |
| --- | --- | --- | --- |
| Factor model | `beta_mkt` | 120d rolling beta to the equal-weight liquid Thai market | Risk exposure; low-beta anomaly |
| | `beta_thb`, `beta_eem` | 120d betas to USD/THB and EEM (lagged) | Sensitivity to foreign flows |
| | `idio_vol` | 60d std of residual return | Idiosyncratic-volatility puzzle; scales alpha in the optimiser |
| | `resid_rev_5` | 5d sum of residual returns | Short-term reversal / liquidity provision |
| | `resid_mom_20`, `resid_mom_120` | 20d and 120d (skip 5d) residual momentum | Residual momentum (Blitz et al.) |
| Liquidity | `log_dvol60` | log median traded value, 60d | Size / liquidity |
| | `vol_z20` | volume z-score vs 20d | Attention / news shocks |
| | `amihud_20` | log(1 + mean abs return per THB traded) | Illiquidity premium |
| | `zero_ret_20` | share of zero-return days | Thin trading |
| | `gap_1`, `range_1` | overnight gap, intraday range / close | Overnight vs intraday information |
| | `max_ret_20` | max daily return in 20d | Lottery-stock (MAX) effect |
| Range | `dist_52w_high`, `dist_52w_low` | distance to 52-week high / low | Anchoring (George & Hwang) |
| | `rel_ret_5`, `rel_ret_20` | return minus market return | Relative strength |
| Ranks | `cs_ret_5`, `cs_ret_20`, `cs_resid_mom_20`, `cs_vol_20`, `cs_dvol` | daily percentile among liquid stocks | Scale-free, robust to regime shifts |
| Thai market | `mkt_ret_1`, `mkt_ret_5`, `breadth`, `dispersion` | market return, share of advancers, cross-sectional std | Market state |
| HMM | `hmm_p_calm`, `hmm_p_stressed` | filtered regime probabilities | Regime-dependent behaviour |
| Foreign flows (proxies) | `usdthb_ret_1/5`, `usdthb_vol_20` | baht moves and volatility | Baht weakness = foreign outflow |
| | `eem_ret_1/5`, `eem_vol_z` | EM ETF return and volume shock | Global EM fund flows |
| Global | `spx_ret_5`, `dxy_ret_5`, `oil_ret_5`, `gold_ret_5` | 5d returns | Risk appetite, commodity exposure |
| Rates / credit | `credit_spread`, `credit_spread_chg_5` | Moody's Baa - 10y (FRED) | Credit stress |
| Cboe | `vix`, `vix_slope` (VIX/VIX3M), `vix9d_ratio`, `vvix`, `skew`, `vix_chg_5` | volatility term structure | Backwardation = stress |

Lecture-style features: returns over 1-120 days, volatility, SMA ratios, RSI, MACD histogram, Bollinger %B, ATR,
US 10y change and 3m T-bill.

In [ ]:
df[sets["NUMERIC"]].describe().T[["count", "mean", "std", "min", "50%", "max"]]

## Generated transformations (binned features -> dummies)

| Feature | Transformation |
| --- | --- |
| `size_q` | quintile of `cs_dvol` (liquidity rank) |
| `mom_q` | quintile of `cs_ret_20` (momentum rank) |
| `rsi_bin` | RSI bucketed into oversold / weak / neutral / strong / overbought |
| `vix_regime` | VIX bucketed into calm / normal / stressed / panic |
| `thb_regime` | 5-day baht move > +1% (weak baht) / < -1% (strong) / neutral |
| `hmm_state` | most likely HMM regime |
| `month`, `weekday`, `sector` | calendar and Yahoo sector |

In [ ]:
print(sets["DUMMIES"][:40], "...")
for c in ["size_q", "mom_q", "rsi_bin", "vix_regime", "thb_regime", "hmm_state"]:
    t = df[df["tradable"] & (df["Date"] < config.VALID_START)]
    print(f"\n{c}: mean 5d alpha (%) by bin (train years)")
    print((t.groupby(c, observed=True)["fwd_alpha_5"].mean() * 100).round(3).to_dict())